# CSEC 303 Midterm Laboratory — Case File PHANTOM-303

**Role:** Digital Imaging Examiner  
**Submission:** Perform the investigations in this notebook, then record your final answers in **Custos**.

You are supporting a medical-imaging and digital-evidence unit. Your task is to recover useful image information, compare objective quality metrics, and justify parameter choices. **You are not being asked to diagnose a patient or identify a real person.**

The Custos assessment is intentionally **open-tool**: switching between Jupyter and Custos is expected. Do not overwrite the source evidence files.

## Case workflow

1. Run the setup cell.
2. Work through Cases 00–04 in order.
3. For each experiment, compare the requested metric/constraint.
4. Answer the corresponding Custos questions.
5. You may rerun cells and change candidate values **only where the case tells you to do so**.

In [ ]:
from pathlib import Path
import io, os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage
from skimage import exposure, filters, metrics, restoration

CASE_DIR = Path('.')
if not (CASE_DIR / 'case00_evidence_original.png').exists():
    CASE_DIR = Path('CSEC303_Midterm_Case_Kit')
if not (CASE_DIR / 'case00_evidence_original.png').exists():
    raise FileNotFoundError('Place this notebook in the extracted CSEC303_Midterm_Case_Kit folder.')

def gray(name):
    return np.asarray(Image.open(CASE_DIR / name).convert('L'), dtype=float) / 255.0

def show(*items, cmap='gray'):
    fig, axes = plt.subplots(1, len(items), figsize=(5*len(items), 5))
    axes = np.atleast_1d(axes)
    for ax, (title, arr) in zip(axes, items):
        ax.imshow(arr, cmap=cmap, vmin=0, vmax=1); ax.set_title(title); ax.axis('off')
    plt.show()

def score(reference, candidate):
    return {'PSNR': metrics.peak_signal_noise_ratio(reference, candidate, data_range=1),
            'SSIM': metrics.structural_similarity(reference, candidate, data_range=1)}

print('Case kit ready:', CASE_DIR.resolve())

## CASE 00 — Evidence Intake, Representation & Compression

Inspect the same synthetic ridge evidence saved in multiple formats. Use the file signatures, file sizes, bit depth concepts, and compression metrics to answer **Custos Q1–Q4**.

For the compression decision, choose the **smallest JPEG quality candidate whose PSNR is at least 32 dB**.

In [ ]:
files = ['case00_evidence_original.png','case00_evidence.tiff','case00_evidence.webp',
         'case00_evidence_q25.jpg','case00_evidence_q45.jpg','case00_evidence_q65.jpg','case00_evidence_q85.jpg']
rows=[]
ref=gray('case00_evidence_original.png')
for name in files:
    p=CASE_DIR/name
    raw=p.read_bytes()
    row={'file':name,'bytes':p.stat().st_size,'signature_hex':raw[:12].hex(' ').upper()}
    if name.endswith('.jpg'):
        cand=gray(name)
        row.update(score(ref,cand))
    rows.append(row)
pd.DataFrame(rows)

## CASE 01 — Medical Image Contrast Recovery

The supplied medical image was intentionally compressed into a low-contrast intensity range. Compare candidate gamma values, then tune CLAHE.

- **Q5:** choose the gamma with the highest SSIM to the reference.
- **Q6:** after using that gamma, choose the CLAHE `clip_limit` whose central ROI mean is **0.49–0.54** and whose ROI standard deviation is **closest to 0.09**.
- **Q7:** compare global histogram equalization and CLAHE using ROI SSIM.
- **Q8:** answer the transformation concept question in Custos.

In [ ]:
ref=gray('case01_medical_reference.png'); low=gray('case01_medical_lowcontrast.png')
show(('Low-contrast input',low),('Reference appearance',ref))

gamma_rows=[]
for g in [0.55,0.75,0.95,1.15]:
    out=exposure.adjust_gamma(low,g)
    gamma_rows.append({'gamma':g, **score(ref,out), 'mean':out.mean()})
pd.DataFrame(gamma_rows)

In [ ]:
# Enter the gamma you selected from the previous table.
selected_gamma = 1.00  # <-- change this
base = exposure.adjust_gamma(low, selected_gamma)
roi=np.s_[96:416,96:416]
rows=[]
for c in [0.005,0.01,0.02,0.04]:
    out=exposure.equalize_adapthist(base,clip_limit=c,kernel_size=64)
    rows.append({'clip_limit':c,'ROI mean':out[roi].mean(),'ROI std':out[roi].std(),
                 'ROI SSIM':metrics.structural_similarity(ref[roi],out[roi],data_range=1)})
pd.DataFrame(rows)

In [ ]:
# Replace the two parameter values below with your chosen gamma and clip limit.
selected_gamma = 1.00
selected_clip = 0.01
base=exposure.adjust_gamma(low,selected_gamma)
he=exposure.equalize_hist(base)
clahe=exposure.equalize_adapthist(base,clip_limit=selected_clip,kernel_size=64)
print('Global HE ROI SSIM:',metrics.structural_similarity(ref[roi],he[roi],data_range=1))
print('CLAHE ROI SSIM:    ',metrics.structural_similarity(ref[roi],clahe[roi],data_range=1))
show(('Global HE',he),('CLAHE',clahe),('Reference',ref))

## CASE 02 — Latent Ridge Evidence: Noise & Spatial Filtering

The ridge pattern is **synthetic** and does not belong to a real person. Compare filters quantitatively.

- **Q9:** salt-and-pepper corruption → highest PSNR.
- **Q10:** Gaussian corruption → highest SSIM.
- **Q11:** median kernel size → highest SSIM for the salt-and-pepper case.
- **Q12:** select the boundary handling method that avoids artificial dark borders.

In [ ]:
clean=gray('case02_fingerprint_clean.png'); sp=gray('case02_fingerprint_saltpepper.png'); gn=gray('case02_fingerprint_gaussian.png')
show(('Clean synthetic ridge',clean),('Salt & pepper',sp),('Gaussian noise',gn))

def compare(noisy, bilateral_color):
    candidates={
      'Mean 3x3':ndimage.uniform_filter(noisy,size=3,mode='reflect'),
      'Gaussian sigma=1':filters.gaussian(noisy,sigma=1,preserve_range=True,mode='reflect'),
      'Median 3x3':filters.median(noisy,footprint=np.ones((3,3)),mode='reflect'),
      'Bilateral':restoration.denoise_bilateral(noisy,sigma_color=bilateral_color,sigma_spatial=3,channel_axis=None)}
    return pd.DataFrame([{'filter':k,**score(clean,v)} for k,v in candidates.items()])

print('Salt-and-pepper candidates')
display(compare(sp,0.12))
print('Gaussian-noise candidates')
display(compare(gn,0.08))

In [ ]:
rows=[]
for k in [3,5,7,9]:
    out=filters.median(sp,footprint=np.ones((k,k)),mode='reflect')
    rows.append({'median kernel':f'{k}x{k}',**score(clean,out)})
pd.DataFrame(rows)

## CASE 03 — Radiographic Phantom: Sharpening & Restoration

A standard Shepp–Logan imaging phantom was blurred and contaminated with Gaussian noise.

- **Q13:** select the unsharp `amount` with the highest PSNR.
- **Q14:** select the Wiener `balance` with the highest PSNR.
- **Q15–Q16:** answer the restoration/sharpening concept questions in Custos.

In [ ]:
ref=gray('case03_phantom_reference.png'); degraded=gray('case03_phantom_degraded.png')
show(('Degraded phantom',degraded),('Reference phantom',ref))
rows=[]
for a in [0.5,1.0,1.5,2.0]:
    out=np.clip(filters.unsharp_mask(degraded,radius=1.5,amount=a,preserve_range=True),0,1)
    rows.append({'amount':a,**score(ref,out)})
pd.DataFrame(rows)

In [ ]:
def gaussian_psf(size=13,sigma=2.0):
    ax=np.arange(-(size//2),size//2+1); xx,yy=np.meshgrid(ax,ax)
    k=np.exp(-(xx**2+yy**2)/(2*sigma**2)); return k/k.sum()
psf=gaussian_psf()
rows=[]
for balance in [0.01,0.03,0.05,0.10]:
    out=restoration.wiener(degraded,psf,balance=balance,clip=True)
    rows.append({'balance':balance,**score(ref,out)})
pd.DataFrame(rows)

## CASE 04 — Periodic Interference: Frequency-Domain Processing

The phantom contains an artificial sinusoidal interference pattern. Examine the centered FFT magnitude and remove the symmetric interference peaks using notch rejection.

- **Q17:** identify the dominant symmetric peak offsets from the spectrum center as `(row offset, column offset)`.
- **Q18:** choose the notch radius with the highest PSNR.
- **Q19–Q20:** answer the frequency-domain concept questions in Custos.

In [ ]:
clean=gray('case04_frequency_clean.png'); noisy=gray('case04_frequency_noisy.png'); spectrum=gray('case04_frequency_spectrum.png')
show(('Periodic interference',noisy),('Log FFT magnitude',spectrum),('Reference',clean))

F=np.fft.fftshift(np.fft.fft2(noisy)); mag=np.abs(F); cy,cx=np.array(mag.shape)//2
# Suppress the central low-frequency neighborhood, then list the strongest remaining peaks.
probe=mag.copy(); Y,X=np.ogrid[:mag.shape[0],:mag.shape[1]]; probe[(Y-cy)**2+(X-cx)**2<15**2]=0
coords=np.argpartition(probe.ravel(),-12)[-12:]
coords=np.array(np.unravel_index(coords,probe.shape)).T
coords=sorted(coords,key=lambda rc:probe[tuple(rc)],reverse=True)
[(int(r-cy),int(c-cx),float(probe[r,c])) for r,c in coords[:8]]

In [ ]:
def notch_filter(img,dy,dx,radius):
    F=np.fft.fftshift(np.fft.fft2(img)); h,w=img.shape; cy,cx=h//2,w//2
    Y,X=np.ogrid[:h,:w]; mask=np.ones((h,w),float)
    for sy,sx in [(dy,dx),(-dy,-dx)]:
        mask[(Y-(cy+sy))**2+(X-(cx+sx))**2<=radius**2]=0
    return np.clip(np.real(np.fft.ifft2(np.fft.ifftshift(F*mask))),0,1)

# Replace dy/dx after reading the peak-offset table above.
dy,dx = 0,0  # <-- change these
rows=[]
for radius in [2,4,6,10]:
    out=notch_filter(noisy,dy,dx,radius)
    rows.append({'radius':radius,**score(clean,out)})
pd.DataFrame(rows)

## Final check

Before submitting Custos, verify that you answered **Q1–Q20**. Keep your notebook as your working paper; only the Custos answers are graded automatically in this local prototype.